# Feature Engineering

This Notebook ingests the parquet file 03_prep_clean.parquet and will be used to create stratified train and test sets ready for modelling. 

In [5]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

sys.path.append("..")

In [6]:
import os
print(os.getcwd())

C:\Users\ashma\projects\grade-profile-ml\notebooks


In [7]:
df = pd.read_parquet(Path("../data/processed/03_prep_clean.parquet"))

In [9]:
print(df.shape)
print(df.columns.tolist())
print(df['level'].value_counts())

(312, 10)
['current_prog', 'level', 'completion_date', 'grade', 'is_distinction', 'epao_AIM', 'epao_AP', 'epao_BCS', 'age_bracket_30-49', 'age_bracket_50+']
level
4    146
3     88
7     78
Name: count, dtype: int64


In [10]:
GRADE_COL = "grade"
LEVEL_COL = "level"
IS_DISTINCTION_COL = "is_distinction"

In [12]:
df_l3 = df[df['level'] == 3]
df_l4 = df[df['level'] == 4]
df_l7 = df[df['level'] == 7]

print(f"L3: {len(df_l3)} rows")
print(f"L4: {len(df_l4)} rows")
print(f"L7: {len(df_l7)} rows")
print(f"Total: {len(df_l3) + len(df_l4) + len(df_l7)} rows")

L3: 88 rows
L4: 146 rows
L7: 78 rows
Total: 312 rows


In [13]:
FEATURE_COLS = [
    'epao_AIM',
    'epao_AP', 
    'epao_BCS',
    'age_bracket_30-49',
    'age_bracket_50+'
]

In [20]:
X_l3 = df_l3[FEATURE_COLS]
y_l3 = df_l3[IS_DISTINCTION_COL]

X_l4 = df_l4[FEATURE_COLS]
y_l4 = df_l4[IS_DISTINCTION_COL]

X_l7 = df_l7[FEATURE_COLS]
y_l7 = df_l7[IS_DISTINCTION_COL]

print(f"L3 — X: {X_l3.shape}, y: {y_l3.shape}")
print(f"L4 — X: {X_l4.shape}, y: {y_l4.shape}")
print(f"L7 — X: {X_l7.shape}, y: {y_l7.shape}")

L3 — X: (88, 5), y: (88,)
L4 — X: (146, 5), y: (146,)
L7 — X: (78, 5), y: (78,)


In [24]:
X_l3_train,X_l3_test,y_l3_train,y_l3_test = train_test_split(X_l3,y_l3,test_size=0.25,train_size=0.75,stratify=y_l3,random_state=42)
X_l4_train,X_l4_test,y_l4_train,y_l4_test = train_test_split(X_l4,y_l4,test_size=0.25,train_size=0.75,stratify=y_l4,random_state=42)
X_l7_train,X_l7_test,y_l7_train,y_l7_test = train_test_split(X_l7,y_l7,test_size=0.25,train_size=0.75,stratify=y_l7,random_state=42)

In [25]:
print(f"L3 — Train: {X_l3_train.shape}, Test: {X_l3_test.shape}")
print(f"L4 — Train: {X_l4_train.shape}, Test: {X_l4_test.shape}")
print(f"L7 — Train: {X_l7_train.shape}, Test: {X_l7_test.shape}")

L3 — Train: (66, 5), Test: (22, 5)
L4 — Train: (109, 5), Test: (37, 5)
L7 — Train: (58, 5), Test: (20, 5)


In [26]:
print(f"L3 — Train distinction rate: {y_l3_train.mean():.2%}, Test: {y_l3_test.mean():.2%}")
print(f"L4 — Train distinction rate: {y_l4_train.mean():.2%}, Test: {y_l4_test.mean():.2%}")
print(f"L7 — Train distinction rate: {y_l7_train.mean():.2%}, Test: {y_l7_test.mean():.2%}")

L3 — Train distinction rate: 62.12%, Test: 63.64%
L4 — Train distinction rate: 52.29%, Test: 51.35%
L7 — Train distinction rate: 6.90%, Test: 5.00%


### Obersvation

The level 7 set has a distinction rate of 5% and with a record count of 20 in the test set for l7 there is a high likelihood of overfitting and the model assigning a non distinction prediction 95% of the time, this is a limitation of the data we have and should be noted

In [28]:
import joblib

splits = {
    'X_l3_train': X_l3_train, 'X_l3_test': X_l3_test,
    'y_l3_train': y_l3_train, 'y_l3_test': y_l3_test,
    'X_l4_train': X_l4_train, 'X_l4_test': X_l4_test,
    'y_l4_train': y_l4_train, 'y_l4_test': y_l4_test,
    'X_l7_train': X_l7_train, 'X_l7_test': X_l7_test,
    'y_l7_train': y_l7_train, 'y_l7_test': y_l7_test,
}

joblib.dump(splits, Path("../data/processed/04_features_splits.joblib"))

['..\\data\\processed\\04_features_splits.joblib']

### Final

This notebook ingested the cleaned data with features and split the data into 3 train and test splits for level 3,4 and 7. FInally exporting the 12 dataframes to a single Joblib file `04_features_splits.joblib`